###Imports, installs & read in data

####installs

In [ ]:
!pip install numpy==1.23.5

In [ ]:
!pip install --force-reinstall spacy thinc

  Using cached spacy-3.8.7-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (27 kB)
  Using cached thinc-9.1.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (14 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.13-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.2 kB)
  Using cached cymem-2.0.11-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.5 kB)
  Using cached preshed-3.0.10-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.4 kB)
  Using cached thinc-8.3.6-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.

#### imports

In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import re
import string
import sys
import nltk
from ast import literal_eval
import time
import math
import numpy as np

####gdrive mount

In [ ]:
from google.colab import drive
drive.mount('drive')

ValueError: mount failed

####downloads

In [ ]:
!pip install https://huggingface.co/sakelariev/bg_news_lg/resolve/main/bg_news_lg-3.5.4-py3-none-any.whl
# !pip install spacy-transformers

####spacy

In [ ]:
import spacy

nlp = spacy.load("bg_news_lg")

#### df load

In [ ]:
df = pd.read_csv(r'drive/My Drive/Colab Notebooks/bg_cds_copyrightsafe.csv')

In [ ]:
df.head(25)

,Source,Utterance,Tokenised,UttLen,MorphTok,def,aorist,imperfect,participle,plural,present
0,https://chitanka.info/book/648-prikazki.txt.zip,Братя Грим,"['братя', 'грим']",2,"['братя', 'грим']",0,0,0,0,0,1
1,https://chitanka.info/book/648-prikazki.txt.zip,Приказки,['приказки'],1,"['приказк', 'и']",0,0,0,0,1,1
2,https://chitanka.info/book/648-prikazki.txt.zip,Умницата Грета,"['умницата', 'грета']",2,"['умница', 'та', 'грета']",1,0,0,0,0,1
3,https://chitanka.info/book/648-prikazki.txt.zip,Имало една готвачка на име Грета.,"['имало', 'една', 'готвачка', 'на', 'име', 'гр...",6,"['има', 'ло', 'едн', 'а', 'готвачка', 'на', 'и...",0,0,0,1,0,1
4,https://chitanka.info/book/648-prikazki.txt.zip,Тя носела обувки с червени токове и като излиз...,"['тя', 'носела', 'обувки', 'с', 'червени', 'то...",23,"['тя', 'носе', 'ла', 'обувки', 'с', 'червен', ...",0,1,1,1,1,1
5,https://chitanka.info/book/648-prikazki.txt.zip,"А щом се върнела вкъщи, още й било толкова вес...","['а', 'щом', 'се', 'върнела', 'вкъщи', 'още', ...",16,"['а', 'щом', 'се', 'върне', 'ла', 'вкъщи', 'ощ...",1,0,0,1,0,1
6,https://chitanka.info/book/648-prikazki.txt.zip,"А тъй като виното събужда желание за ядене, по...","['а', 'тъй', 'като', 'виното', 'събужда', 'жел...",20,"['а', 'тъй', 'като', 'вино', 'то', 'събужд', '...",1,0,0,1,1,1
7,https://chitanka.info/book/648-prikazki.txt.zip,И си казвала:,"['и', 'си', 'казвала']",3,"['и', 'си', 'казва', 'ла']",0,0,0,1,0,1
8,https://chitanka.info/book/648-prikazki.txt.zip,— Всяка готвачка трябва да знае вкуса на гозби...,"['всяка', 'готвачка', 'трябва', 'да', 'знае', ...",9,"['всяк', 'а', 'готвачка', 'трябва', 'да', 'зна...",1,0,0,0,1,1
9,https://chitanka.info/book/648-prikazki.txt.zip,Веднъж господарят й рекъл:,"['веднъж', 'господарят', 'й', 'рекъл']",4,"['веднъж', 'господар', 'ят', 'й', 'рекъ', 'л']",1,0,0,1,0,1


In [ ]:
df.size

612150

###Utils

In [ ]:
def save_sents(df, column_name, filepath):
  sents = [" ".join(literal_eval(u))+"\n" for u in df[column_name]]

  with open(filepath, mode="w") as f:
    f.writelines(sents)

### Save some sents for training

In [ ]:
# save_sents(df, "Tokenised", "drive/My Drive/Colab Notebooks/sents_normtok_bg_cds_copyrightsafe.raw" )

In [ ]:
save_sents(df, "MorphTok", "drive/My Drive/Colab Notebooks/sents_morphtok_bg_cds_copyrightsafe.raw" )

##Morph tok

In [ ]:
ARTICLE_SUFFIXES = ["ът", "ят", "та", "то", "те", "а", "я",]

def is_definite(token):
    #spaCy’s morphological analysis marks this token as definite
    return token.morph.get("Definite") == ["Def"]

def split_token(token):
    """
    Given a spaCy token, return a list of morpheme-tokens:
      - If not definite: [lowercased form], flag 0
      - If definite: [lemma, article_suffix], flag 1
    """
    if not is_definite(token):
        return [token.lower_], 0

    text = token.lower_
    lemma = token.lemma_.lower()
    # find which suffix matches
    for suffix in ARTICLE_SUFFIXES:
        if text.endswith(suffix):
            if lemma.endswith(suffix):
              lemma = lemma[:-len(suffix)]
            return [lemma, suffix], 1

    # fallback: no known suffix found, but morph says Definite
    # treat entire token as stem
    return [lemma], 1

def process_sentences(sents):
    """
    Process a list of raw sentences into:
      - sents_morphtok: list of lists of morpheme-tokens
      - definite_flags: flat list of 0/1 flags in token order
    """
    sents_morphtok = []
    definite_flags =[ ]

    for sent_text in sents:
        doc = nlp(sent_text)
        morphtok_sent = []
        sent_flag = 0
        for token in doc:
            if token.is_punct:
                continue
            morphemes, flag = split_token(token)
            morphtok_sent.extend(morphemes)
            if flag:
                sent_flag = 1
        definite_flags.append(sent_flag)
        sents_morphtok.append(morphtok_sent)

    return sents_morphtok, definite_flags

#### version using spacy morph

In [ ]:
ARTICLE_SUFFIXES = ["ът", "ят", "та", "то", "те", "а", "я"]
AORIST_SUFFIXES = ["х", "хме", "хте", "ха"]
IMPERFECT_SUFFIXES = ["х", "ше", "хме", "хте", "ха"]
PARTICIPLE_SUFFIXES = ["л", "ла", "ло", "ли"]
PRESENT_SING_SUFFIXES = ["а", "я", "еш", "иш", "е", "и"]
PRESENT_PLUR_SUFFIXES = ["ем", "им", "ете", "ите", "ат", "ят"]
NOUN_PLURAL_SUFFIXES = ["и", "ове", "е", "еве", "а", "ета"]

# Utilities
def ends_in_suffix(token, suffixes):
    text = token.lower_
    for suffix in sorted(suffixes, key=lambda s: -len(s)):
        if text.endswith(suffix) and len(text) > len(suffix):
            return suffix
    return None

# Feature Checkers
def is_definite(token):
    return token.morph.get("Definite") == ["Def"]

def is_aorist(token):
    return ends_in_suffix(token, AORIST_SUFFIXES) is not None

def is_imperfect(token):
    return ends_in_suffix(token, IMPERFECT_SUFFIXES) is not None

def is_participle(token):
    return ends_in_suffix(token, PARTICIPLE_SUFFIXES) is not None

def is_plural_noun(token):
    return "Number=Plur" in token.morph and token.pos_ == "NOUN"

def is_present(token):
    return ends_in_suffix(token, PRESENT_SING_SUFFIXES + PRESENT_PLUR_SUFFIXES) is not None

# Token Splitter
def split_token(token):
    text = token.lower_

    # Check each morpheme type
    if is_definite(token):
        suffix = ends_in_suffix(token, ARTICLE_SUFFIXES)
        if suffix:
            return [text[:-len(suffix)], suffix], 1

    for check_fn, suffixes in [
        (is_aorist, AORIST_SUFFIXES),
        (is_imperfect, IMPERFECT_SUFFIXES),
        (is_participle, PARTICIPLE_SUFFIXES),
        (is_present, PRESENT_SING_SUFFIXES + PRESENT_PLUR_SUFFIXES),
        (is_plural_noun, NOUN_PLURAL_SUFFIXES)
    ]:
        if check_fn(token):
            suffix = ends_in_suffix(token, suffixes)
            if suffix:
                return [text[:-len(suffix)], suffix], 1

    return [text], 0

# Sentence Processor
def process_sentences(sents):
    sents_morphtok = []
    definite_flags = []

    for sent_text in sents:
        doc = nlp(sent_text)
        morphtok_sent = []
        sent_flag = 0
        for token in doc:
            if token.is_punct:
                continue
            morphemes, flag = split_token(token)
            morphtok_sent.extend(morphemes)
            if flag:
                sent_flag = 1
        sents_morphtok.append(morphtok_sent)
        definite_flags.append(sent_flag)

    return sents_morphtok, definite_flags

####version using bg specific labels

In [ ]:
def is_definite(token):
    return token.morph.get("Definite") == ["Def"]

def is_imperfect(token):
    return token.tag_.startswith("V") and len(token.tag_) > 2 and token.tag_[2] == "i"

def is_aorist(token):
    return token.tag_.startswith("V") and len(token.tag_) > 2 and token.tag_[2] == "a"

def is_present(token):
    return token.tag_.startswith("V") and len(token.tag_) > 2 and token.tag_[2] == "t"

def is_participle(token):
    return token.tag_.startswith("V") and len(token.tag_) > 4 and token.tag_[4] == "p"

def is_plural_noun(token):
    return token.tag_.startswith("N") and len(token.tag_) > 7 and token.tag_[7] == "p"

def split_token(token):
    text = token.lower_
    lemma = token.lemma_.lower()

    # If lemma is in exclusion list, return as-is
    if lemma in EXCLUDED_LEMMAS:
        return [text], 0

    # Try definite article split
    if is_definite(token):
        suffix = ends_in_suffix(token, ARTICLE_SUFFIXES)
        if suffix:
            return [text[:-len(suffix)], suffix], 1

    # No suffix split for other morphs
    if any([
        is_imperfect(token),
        is_aorist(token),
        is_participle(token),
        is_present(token),
        is_plural_noun(token)
    ]):
        return [text], 1

    return [text], 0

def process_sentences(sents):
    morphtok_all = []
    def_flags = []
    aorist_flags = []
    imperfect_flags = []
    part_flags = []
    plur_flags = []
    pres_flags = []

    for sent_text in sents:
        doc = nlp(sent_text)
        tokens = [t for t in doc if not t.is_punct]
        morphtok_sent = []
        flags = {
            "def": 0, "aorist": 0, "imperfect": 0,
            "participle": 0, "plural": 0, "present": 0
        }

        for token in tokens:
            morphemes, _ = split_token(token)
            morphtok_sent.extend(morphemes)

            if is_definite(token): flags["def"] = 1
            if is_aorist(token): flags["aorist"] = 1
            if is_imperfect(token): flags["imperfect"] = 1
            if is_participle(token): flags["participle"] = 1
            if is_plural_noun(token): flags["plural"] = 1
            if is_present(token): flags["present"] = 1

        morphtok_all.append(morphtok_sent)
        def_flags.append(flags["def"])
        aorist_flags.append(flags["aorist"])
        imperfect_flags.append(flags["imperfect"])
        part_flags.append(flags["participle"])
        plur_flags.append(flags["plural"])
        pres_flags.append(flags["present"])

    return (
        morphtok_all, def_flags, aorist_flags,
        imperfect_flags, part_flags, plur_flags, pres_flags
    )

#### hacky version

In [ ]:
ARTICLE_SUFFIXES = ["ът", "ят", "та", "то",  "те", "а", "я"] # "ите","ия",
AORIST_SUFFIXES = ["х", "хме", "хте", "ха"]
IMPERFECT_SUFFIXES = ["х", "ше", "хме", "хте", "ха"]
PARTICIPLE_SUFFIXES = [ "ла", "ло", "ли", "л",]
PRESENT_SING_SUFFIXES = [ "еш", "иш", "е", "и", "а", "я",]
PRESENT_PLUR_SUFFIXES = ["ме", "ем", "им", "ете", "ите", "ат", "ят", ]
NOUN_PLURAL_SUFFIXES = ["ове", "еве", "ета", "а","е", "и",]

# Utilities
def ends_in_suffix(token, suffixes):
    text = token.lower_
    for suffix in suffixes:
    # for suffix in sorted(suffixes, key=lambda s: -len(s)):
        if text.endswith(suffix) and len(text) > len(suffix):
            return suffix
    return None

# Feature Checkers
def is_definite(token):
    return token.morph.get("Definite") == ["Def"]

def is_aorist(token):
    return ends_in_suffix(token, AORIST_SUFFIXES) is not None

def is_imperfect(token):
    return ends_in_suffix(token, IMPERFECT_SUFFIXES) is not None

def is_participle(token):
    return ends_in_suffix(token, PARTICIPLE_SUFFIXES) is not None

def is_plural_noun(token):
    return "Number=Plur" in token.morph and token.pos_ == "NOUN"

def is_present(token):
    return ends_in_suffix(token, PRESENT_SING_SUFFIXES + PRESENT_PLUR_SUFFIXES) is not None

def is_nouny(token):
      return token.pos_ in {"NOUN", "ADJ", "PROPN", "PRON"}


# Token Splitter
def split_token(token):
    text = token.lower_
    lemma = token.lemma_.lower()

    # If lemma is in exclusion list, return as-is
    if text in EXCLUDED_LEMMAS:
        # print("in here")
        return [text], 0

    # Check each morpheme type
    if is_definite(token) or is_nouny(token):
        suffix = ends_in_suffix(token, ARTICLE_SUFFIXES)
        # print(token, suffix)
        if suffix:
            return [text[:-len(suffix)], suffix], 1

    for check_fn, suffixes in [
        (is_aorist, AORIST_SUFFIXES),
        (is_imperfect, IMPERFECT_SUFFIXES),
        (is_participle, PARTICIPLE_SUFFIXES),
        (is_present, PRESENT_PLUR_SUFFIXES + PRESENT_SING_SUFFIXES ),
        (is_plural_noun, NOUN_PLURAL_SUFFIXES)
    ]:
        if check_fn(token):
            suffix = ends_in_suffix(token, suffixes)
            if suffix:
                return [text[:-len(suffix)], suffix], 1

    return [text], 0

def process_sentence(sent_text):
    """
    Process a single sentence and return:
    - morpheme-tokenized list
    - flags for morph categories
    """
    flags = {
        "def": 0, "aorist": 0, "imperfect": 0,
        "participle": 0, "plural": 0, "present": 0
    }
    morphtok_sent = []
    doc = nlp(sent_text)

    for token in doc:
        if token.is_punct:
            continue
        morphemes, _ = split_token(token)
        morphtok_sent.extend(morphemes)

        if is_definite(token): flags["def"] = 1
        if is_aorist(token): flags["aorist"] = 1
        if is_imperfect(token): flags["imperfect"] = 1
        if is_participle(token): flags["participle"] = 1
        if is_plural_noun(token): flags["plural"] = 1
        if is_present(token): flags["present"] = 1

    return morphtok_sent, flags
def process_sentences(sents):
    """
    Process a list of sentences using `process_sentence`
    """
    sents_morphtok = []
    def_flags = []
    aorist_flags = []
    imperfect_flags = []
    part_flags = []
    plur_flags = []
    pres_flags = []

    for sent_text in sents:
        morphtok_sent, flags = process_sentence(sent_text)

        sents_morphtok.append(morphtok_sent)
        def_flags.append(flags["def"])
        aorist_flags.append(flags["aorist"])
        imperfect_flags.append(flags["imperfect"])
        part_flags.append(flags["participle"])
        plur_flags.append(flags["plural"])
        pres_flags.append(flags["present"])

    return (
        sents_morphtok, def_flags, aorist_flags,
        imperfect_flags, part_flags, plur_flags, pres_flags
    )


In [ ]:
# assert(is_definite(nlp(test_sents[0])[1]))
# assert(not is_definite(nlp(test_sents[0])[2]))

test_sents = [
    "Виждахме мечката и кучето в градината.",  # imperfect: "виждахме", article: "мечката"
    "Те говориха много.",                      # aorist: "говориха"
    "Децата играят в парка.",                  # plural article: "децата", verb: "играят"
    "Бях гледал филма.",                       # participle: "гледал"
    "Умницата Грета",                          # article: "умницата"
    "Котките са жълти.",                        # definite (plural)
    "По-хубаво е в София.",                     # comparative
    "по-хубаво",
    "ядрената"
]

process_sentences(test_sents)


([['вижда', 'хме', 'мечка', 'та', 'и', 'куче', 'то', 'в', 'градина', 'та'],
  ['те', 'говори', 'ха', 'много'],
  ['деца', 'та', 'игра', 'ят', 'в', 'парка'],
  ['бя', 'х', 'гледа', 'л', 'филм', 'а'],
  ['умница', 'та', 'грета'],
  ['котки', 'те', 'с', 'а', 'жълти'],
  ['по', 'хубаво', 'е', 'в', 'софия'],
  ['по', 'хубаво'],
  ['ядрена', 'та']],
 [1, 0, 1, 1, 1, 1, 0, 0, 1],
 [1, 1, 0, 1, 0, 0, 0, 0, 0],
 [1, 1, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 1, 0, 0, 0],
 [1, 1, 1, 1, 1, 1, 1, 0, 1])

In [ ]:
(
    df["MorphTok"], df["def"], df["aorist"],
    df["imperfect"], df["participle"], df["plural"], df["present"]
) = process_sentences(df["Utterance"].tolist())

KeyboardInterrupt: 

In [ ]:
# sents_morphtok, definite = process_sentences(df.Utterance.tolist())
# df["MorphTok"] = sents_morphtok
# df["def"] = definite

In [ ]:
df.to_csv(r'drive/My Drive/Colab Notebooks/bg_cds_copyrightsafe.csv', index=False)

In [ ]:
df.MorphTok.tolist()

###BG Morphemes

In [ ]:
# Aorist (Минало свършено време)
aorist_endings = [
    "-х",   # 1 л. ед.ч.
    "",     # 2 л. ед.ч.
    "",     # 3 л. ед.ч.
    "-хме", # 1 л. мн.ч.
    "-хте", # 2 л. мн.ч.
    "-ха"   # 3 л. мн.ч.
]

# Imperfect (Минало несвършено време)
imperfect_endings = [
    "-х",   # 1 л. ед.ч.
    "-ше",  # 2 л. ед.ч.
    "-ше",  # 3 л. ед.ч.
    "-хме", # 1 л. мн.ч.
    "-хте", # 2 л. мн.ч.
    "-ха"   # 3 л. мн.ч.
]

# Perfect (Минало неопределено време)
# Each entry is a tuple: (masc, fem, plural)
participle_endings = ["-л", "-ла", "-ло", "-ли"]

plural_noun_endings = {
    "masculine": {
        "hard consonant": ["-и", "-ове", "-е"],
        "soft consonant (й, ч, ш)": ["-еве", "-ове", "-и"]
    },
    "feminine": {
        "ends in -а / -я": ["-и"],
        "ends in consonant": ["-и", "-ове"]
    },
    "neuter": {
        "ends in -о / -е": ["-а", "-ета"],
        "ends in stressed -е": ["-ета"]
    }
}

# Singular endings for present tense
present_singular_endings = [
    "-а", "-я",       # 1st person singular
    "-еш", "-иш",     # 2nd person singular
    "-е", "-и"        # 3rd person singular
]

# Plural endings for present tense
present_plural_endings = [
    "-ем", "-им",             # 1st person plural
    "-ете", "-ите",           # 2nd person plural
    "-ат", "-ят"              # 3rd person plural
]

###Exceptions

In [ ]:
import requests
import gzip
import io
from urllib.parse import unquote


SITEMAP_URLS = [
    "http://rechnik.chitanka.info/sitemaps/word.1.txt.gz",
    "http://rechnik.chitanka.info/sitemaps/word.2.txt.gz",
    "http://rechnik.chitanka.info/sitemaps/word.3.txt.gz",
]

EXCLUDED_LEMMAS = []

for url in SITEMAP_URLS:
    print(f"Downloading: {url}")
    response = requests.get(url)
    with gzip.open(io.BytesIO(response.content), 'rt', encoding='utf-8') as f:
        for line in f:
            EXCLUDED_LEMMAS.append(line.strip())

# Save to file (optional)
# with open("lemmas.txt", "w", encoding="utf-8") as out:
#     for lemma_url in EXCLUDED_LEMMAS:
#         out.write(lemma_url + "\n")

print(f"Extracted {len(EXCLUDED_LEMMAS)} lemmas.")

EXCLUDED_LEMMAS = [unquote(url.split("/w/")[1]) for url in EXCLUDED_LEMMAS]
EXCLUDED_LEMMAS = [e.lower() for e in EXCLUDED_LEMMAS]
EXCLUDED_LEMMAS = set(EXCLUDED_LEMMAS)

Downloading: http://rechnik.chitanka.info/sitemaps/word.1.txt.gz
Downloading: http://rechnik.chitanka.info/sitemaps/word.2.txt.gz
Downloading: http://rechnik.chitanka.info/sitemaps/word.3.txt.gz
Extracted 117871 lemmas.


words that end in та

In [ ]:
endings = {}

In [ ]:
w_ta = [w for w in words if w.endswith("та")]

NameError: name 'words' is not defined

In [ ]:
w_i = [w for w in words if w.endswith("и")]

In [ ]:
for s in nlp("Вървях си под моста и гледам в дола момиченце малко печеше цаца. Четох, че рибите са сини. Чиниите ръмжаха силно. Виждахме мечката и кучето в градината. Те говориха много. Децата играят в парка. Бях гледал филма. Умницата Грета. ").sents:
  print('\n')
  for t in s:
    if t.pos_ != "PUNCT":
      print(t.lower_, t.lemma_, t.pos_, t.tag_, t.morph) #doesn't have: t.lex_, t.morph_

## MorphScore


#### score for bg is 0.584 from https://aclanthology.org/2025.coling-main.441.pdf and

In [ ]:
# --- Step 1: Imports ---
import pandas as pd
import numpy as np
import requests
import io

# --- Step 2: Load Bulgarian MorphScore Data from GitHub ---
url = "https://github.com/catherinearnett/morphscore/raw/v1/data/bulgarian_morph_data.csv"
csv_data = requests.get(url).content
morph_data = pd.read_csv(io.StringIO(csv_data.decode("utf-8")))

# --- Step 3: MorphScore Evaluation Logic ---
def morph_eval(morphemes, tokens):  # returns -1, 0, 1
    if len(tokens) == 1:
        return 0
    for t in range(len(tokens)-1):
        pt1 = ''.join(tokens[:t+1])
        rest = ''.join(tokens[t+1:])
        if [pt1, rest] == morphemes:
            return 1
    return -1

def get_morphscore(data, tokenizer):
    points = []
    error_rows = []

    for _, row in data.iterrows():
        morphemes = [row['pt1'], row['rest']]
        word = row['full_word']
        tokens = tokenizer(word)

        point = morph_eval(morphemes, tokens)

        # Only score cases with a segmentation attempt
        if point != 0:
            points.append(0 if point == -1 else 1)

        # Collect error info if incorrect
        if point == -1:
            error_rows.append({
                "full_word": word,
                "gold": morphemes,
                "predicted": tokens,
                "result": "wrong"
            })
        elif point == 1:
            error_rows.append({
                "full_word": word,
                "gold": morphemes,
                "predicted": tokens,
                "result": "correct"
            })

    morph_score = np.mean(points) if points else 0.0
    error_df = pd.DataFrame(error_rows)
    return morph_score, error_df

In [ ]:
def my_tokenizer(sentence):
    morphemes, _ = process_sentence(sentence)
    return morphemes

# Now use it with get_morphscore
score, errors_df = get_morphscore(morph_data, my_tokenizer)

print("MorphScore:", round(score, 4))
display(errors_df)  # If using Jupyter

MorphScore: 0.6996


,full_word,gold,predicted,result
0,вятърът,"[вятър, ът]","[вятър, ът]",correct
1,ураганът,"[ураган, ът]","[ураган, ът]",correct
2,посланическите,"[посланически, те]","[посланически, те]",correct
3,затвора,"[затвор, а]","[затвор, а]",correct
4,приемат,"[приема, т]","[прием, ат]",wrong
...,...,...,...,...
1899,топлофикацията,"[топлофикация, та]","[топлофикация, та]",correct
1900,целта,"[цел, та]","[цел, та]",correct
1901,брокерите,"[брокер, ите]","[брокери, те]",wrong
1902,русалката,"[русалка, та]","[русалка, та]",correct


In [ ]:
errors_df[errors_df.result=="wrong"].head(50)

,full_word,gold,predicted,result
4,приемат,"[приема, т]","[прием, ат]",wrong
14,ядрената,"[ядрен, ата]","[ядрена, та]",wrong
17,изборите,"[избор, ите]","[избори, те]",wrong
24,нарядите,"[наряд, ите]","[наряди, те]",wrong
26,широките,"[широк, ите]","[широки, те]",wrong
27,черните,"[чер, ните]","[черни, те]",wrong
29,веселата,"[весел, ата]","[весела, та]",wrong
35,страдаме,"[страдам, е]","[страда, ме]",wrong
41,респондентите,"[респондент, ите]","[респонденти, те]",wrong
42,спринта,"[спринт, а]","[сприн, та]",wrong


### Discarded: getting lemmas from the BTB, but they are not actually all lemmas and only 14K

In [ ]:
import os
import requests
import zipfile

# Step 1: Download and unzip UD Bulgarian-BTB dataset
url = "https://github.com/UniversalDependencies/UD_Bulgarian-BTB/archive/refs/tags/r2.12.zip"
zip_path = "bg_btb.zip"

print("⬇️ Downloading UD Bulgarian Treebank...")
r = requests.get(url)
with open(zip_path, "wb") as f:
    f.write(r.content)

print("🗂️ Extracting...")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(".")

base_folder = "UD_Bulgarian-BTB-r2.12"
conllu_files = [
    "bg_btb-ud-train.conllu",
    "bg_btb-ud-dev.conllu",
    "bg_btb-ud-test.conllu"
]

# Step 2: Parse lemmas
lemmas = set()

for file in conllu_files:
    path = os.path.join(base_folder, file)
    with open(path, encoding="utf-8") as f:
        for line in f:
            print(line)
            if line.strip() and not line.startswith("#"):
                fields = line.split("\t")
                if len(fields) >= 3:
                    lemma = fields[2]
                    if lemma != "_":
                        lemmas.add(lemma)

print(f"✅ Extracted {len(lemmas)} unique lemmas")

# Step 3: Save lemmas to file
lemmas_file = "bulgarian_lemmas.txt"
with open(lemmas_file, "w", encoding="utf-8") as f:
    for lemma in sorted(lemmas):
        f.write(lemma + "\n")

print(f"📁 Lemmas saved to {lemmas_file}")


Streaming output truncated to the last 5000 lines.
# text = Глобата за продажба на алкохол и цигари без лиценз е 15000 лв.

1	Глобата	глоба	NOUN	Ncfsd	Definite=Def|Gender=Fem|Number=Sing	12	nsubj	12:nsubj	_

2	за	за	ADP	R	_	3	case	3:case	_

3	продажба	продажба	NOUN	Ncfsi	Definite=Ind|Gender=Fem|Number=Sing	1	nmod	1:nmod:за	_

4	на	на	ADP	R	_	5	case	5:case	_

5	алкохол	алкохол	NOUN	Ncmsi	Definite=Ind|Gender=Masc|Number=Sing	3	nmod	3:nmod:на	_

6	и	и	CCONJ	Cp	_	7	cc	7:cc	_

7	цигари	цигара	NOUN	Ncfpi	Definite=Ind|Gender=Fem|Number=Plur	5	conj	5:conj	_

8	без	без	ADP	R	_	9	case	9:case	_

9	лиценз	лиценз	NOUN	Ncmsi	Definite=Ind|Gender=Masc|Number=Sing	3	nmod	3:nmod:без	_

10	е	съм	AUX	Vxitf-r3s	Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin|Voice=Act	12	cop	12:cop	_

11	15000	15000	NUM	Mc-pi	Definite=Ind|Number=Plur|NumType=Card	12	nummod	12:nummod	_

12	лв.	лев	NOUN	Ncmt	Gender=Masc|Number=Count	0	root	0:root	_



# sent_id = Standard-2000-10-24-s200

# text = Премиерът 

In [ ]:
lemmas

{'ултранационалистически',
 'придобивка',
 'настойчив',
 'бламирам',
 'динков',
 'иванкович',
 'смут',
 'поръмжа',
 'дванайсети',
 'противоположно',
 'бижу',
 'кацарова',
 'цветомир',
 'строя',
 'обвързвам-(се)',
 'повдигна-(се)',
 'вадя',
 'оркестър',
 'предубеждение',
 'обяд',
 'ататюрк',
 'отвред',
 'ниско',
 'левон',
 'виена',
 'алинея',
 'благородство',
 'човечество',
 '7000',
 'технически',
 'ненакърним',
 'съществено',
 'авторски',
 'престол',
 'непосилен',
 'молебен',
 'дете',
 'цялостност',
 'царуване',
 'новосибирск',
 '1987/1988',
 'учредителен',
 'маестро',
 'проектиране',
 'галин',
 'кастро',
 'нсбоп',
 'лорд',
 'булка',
 'инокентий',
 'разрез',
 'извия-(се)',
 'словения',
 'газстроймонтаж',
 'бачковски',
 'някъде',
 'шломо',
 'aes',
 'паркиране',
 'стремеж',
 'права',
 'мотор-институция',
 'задържа-(се)',
 'шетам',
 'акага',
 'следа',
 'занаят',
 'терор',
 'машина',
 'тежък',
 'bednar',
 'лила',
 '1995',
 'киносценарий',
 'огрея',
 'потъмнея',
 'доказване',
 'зачаквам',
 

In [ ]:
ta_lemmas = [lemma for lemma in lemmas if lemma.endswith("та")]

In [ ]:
ta_lemmas

['чета',
 'леснота',
 'цефта',
 'мента',
 'касета',
 'малта',
 'черта',
 'коста',
 'траволта',
 'координата',
 'заплата',
 'предпочета',
 '37-ата',
 'нафта',
 'минута',
 'работа',
 '53-ата',
 'анита',
 '28-годишната',
 'джакарта',
 'невеста',
 'санта',
 '21-та',
 'белота',
 'срамота',
 'чистота',
 '27-годишната',
 'харта',
 'валета',
 'оферта',
 'яхта',
 'карта',
 'регата',
 '5-ата',
 'помета',
 'порта',
 'прочета',
 'визита',
 '89-ата',
 'та',
 '74-ата',
 'мечта',
 'никита',
 'афродита',
 'пета',
 'ранглиста',
 'маргита',
 'марта',
 'атланта',
 'артфиеста',
 'делта',
 'рмд-та',
 'пшеничката',
 '52-та',
 'лента',
 'уста',
 'щафета',
 'галата',
 '65-годишната',
 'лопата',
 'висота',
 '43-годишната',
 'лепта',
 '22-годишната',
 'дочета',
 'доста',
 'писта',
 'монета',
 'лаута',
 'диета',
 'рецепта',
 'салата',
 'херта',
 'охота',
 'субота',
 'анкета',
 'палата',
 '44-годишната',
 'бета',
 'пустота',
 'бта',
 'орбита',
 'просвета',
 '45-часовата',
 'защита',
 'двеста',
 'вендета',
 '22-ра